In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_"

In [111]:
!pip install langchain chromadb huggingface tiktoken pypdf langchain_huggingface langchain-community

In [112]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

In [113]:
embedding=HuggingFaceEndpointEmbeddings(
    model='all-MiniLM-L6-v2'
)

### Create LangChain documents for IPL players

In [114]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [115]:
docs = [doc1, doc2, doc3, doc4, doc5]


### Vector Database

In [125]:
vector_stores=Chroma(
    embedding_function=HuggingFaceEndpointEmbeddings(),
    persist_directory='./chroma_DataBase',
    collection_name='sample'
)

#### add the document in the vector database

In [126]:
vector_stores.add_documents(docs)

['87293edb-76da-4cf4-84e2-1dcfe619810e',
 '13b47071-3239-436b-91f7-7ddd171d04b8',
 'a4923c8e-a1c0-4b0f-a193-552d2f585d69',
 '02b26649-e279-4511-81fb-8a46f75c944d',
 '4259883c-c918-443f-9443-c456eff2a1f0']

#### view documents

In [127]:
vector_stores.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['87293edb-76da-4cf4-84e2-1dcfe619810e',
  '13b47071-3239-436b-91f7-7ddd171d04b8',
  'a4923c8e-a1c0-4b0f-a193-552d2f585d69',
  '02b26649-e279-4511-81fb-8a46f75c944d',
  '4259883c-c918-443f-9443-c456eff2a1f0'],
 'embeddings': array([[-0.03964755, -0.0080165 , -0.01412728, ...,  0.06814161,
         -0.00116094, -0.02425428],
        [-0.01992598,  0.00161884, -0.00959342, ...,  0.02433112,
         -0.003414  , -0.01722031],
        [-0.04256172,  0.01060326, -0.01305765, ...,  0.04738368,
         -0.017487  ,  0.00204661],
        [-0.02724828, -0.03051382,  0.01804127, ...,  0.01937736,
          0.01266331, -0.00295011],
        [ 0.02301698, -0.02011725, -0.00123579, ...,  0.02533474,
          0.00280538, -0.0512028 ]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

#### search Documents

In [128]:
vector_stores.similarity_search(
    query='who among these are a bowler',
    k=1
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

#### search with similarity score

In [129]:
vector_stores.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.185790777206421),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.2036770582199097)]

#### meta-data filtering

In [130]:
vector_stores.similarity_search_with_score(
    query="",
    filter={'team': "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.887956976890564),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  2.0017194747924805)]

#### update documents

In [131]:
updated_doc = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_stores.update_document(document_id='df180544-4db1-4b28-96c6-55b109399bbf', document=updated_doc)

In [132]:
vector_stores.get(include=['documents'])

{'ids': ['87293edb-76da-4cf4-84e2-1dcfe619810e',
  '13b47071-3239-436b-91f7-7ddd171d04b8',
  'a4923c8e-a1c0-4b0f-a193-552d2f585d69',
  '02b26649-e279-4511-81fb-8a46f75c944d',
  '4259883c-c918-443f-9443-c456eff2a1f0'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic all-rou

#### delete document

In [133]:
vector_stores.delete(ids=['7b08b2f0-c858-4354-ab40-40b7aed938a7'])
vector_stores.get(include=['documents'])

{'ids': ['87293edb-76da-4cf4-84e2-1dcfe619810e',
  '13b47071-3239-436b-91f7-7ddd171d04b8',
  'a4923c8e-a1c0-4b0f-a193-552d2f585d69',
  '02b26649-e279-4511-81fb-8a46f75c944d',
  '4259883c-c918-443f-9443-c456eff2a1f0'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic all-rou